# 4-Class Defect Detection — YOLO26 (Nano & Small)

This notebook implements 3D printer defect detection using YOLO26 (Nano and Small classification variants) on a 4-class dataset.

The dataset is loaded from the pre-split directories in `splitted data/` containing `train`, `val`, and `test` splits.

The training parameters, augmentations, and evaluation metrics match the setup used for the MobileNet models, adapted for the Ultralytics YOLO API.

---
## 1. Imports & Configuration

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from ultralytics import YOLO

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)

# Seed for reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 4
CLASS_NAMES = ['Cracking', 'Stringing', 'Warping', 'No_Defect']

print('Imports completed. Ultralytics YOLO26 is ready.')
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

2026-05-21 17:12:31.405827: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779379951.448740   69574 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779379951.463262   69574 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-21 17:12:31.553930: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Imports completed. Ultralytics YOLO26 is ready.
TensorFlow version: 2.18.0
GPU Available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


---
## 2. Load Dataset Paths for Evaluation

We load image paths and labels from `splitted data/` folder directly. This is used later to run manual inference using sklearn metrics.

In [2]:
BASE = os.getcwd()
SPLITTED_DATA_DIR = os.path.join(BASE, 'splitted data')

train_dir = os.path.join(SPLITTED_DATA_DIR, 'train')
val_dir = os.path.join(SPLITTED_DATA_DIR, 'val')
test_dir = os.path.join(SPLITTED_DATA_DIR, 'test')

def get_paths_and_labels(folder_path):
    paths = []
    labels = []
    class_map = {name: idx for idx, name in enumerate(CLASS_NAMES)}
    for class_name in CLASS_NAMES:
        class_dir = os.path.join(folder_path, class_name)
        if os.path.isdir(class_dir):
            for f in os.listdir(class_dir):
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    paths.append(os.path.join(class_dir, f))
                    labels.append(class_map[class_name])
    return np.array(paths), np.array(labels)

train_paths, train_labels = get_paths_and_labels(train_dir)
val_paths, val_labels = get_paths_and_labels(val_dir)
test_paths, test_labels = get_paths_and_labels(test_dir)

print('Dataset split summary:')
print(f'  Train : {len(train_paths)} images')
print(f'  Val   : {len(val_paths)} images')
print(f'  Test  : {len(test_paths)} images')

Dataset split summary:
  Train : 6187 images
  Val   : 773 images
  Test  : 776 images


---
# 3. YOLO26n-cls Model Training & Evaluation

In [3]:
# Step 1 & 2: Load pretrained base model
model_n = YOLO('yolo26n-cls.pt')

# Freeze all layers of the base model except classification head
num_layers_n = len(list(model_n.model.model.children()))
freeze_layers_n = num_layers_n - 1

print(f'Total layers: {num_layers_n}')
print(f'Layers to freeze (backbone): {freeze_layers_n}')

Total layers: 11
Layers to freeze (backbone): 10


### Step 4: YOLO26n-cls Phase 1 — Frozen Base Training (`lr0 = 1e-4`)

In [ ]:
# Train model with frozen backbone
model_n.train(
    data=SPLITTED_DATA_DIR,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    lr0=1e-4,
    optimizer='Adam',
    freeze=freeze_layers_n,
    project='yolo26n_4class',
    name='phase1',
    exist_ok=True,
    seed=SEED,
    fliplr=0.5,
    flipud=0.5,
    degrees=20.0,
    hsv_v=0.2
)
print('✅ YOLO26n-cls Phase 1 Complete.')

New https://pypi.org/project/ultralytics/8.4.52 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.51 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 2060 SUPER, 8192MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/feriel/PFE/Defect-Detection/splitted data, degrees=20.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.2, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-cls.pt, momentum=0

### Step 5: YOLO26n-cls Phase 2 — Fine-Tuning (Unfreeze Last 20 Layers)

In [ ]:
# Load Phase-1 best weights
model_n_ft = YOLO('yolo26n_4class/phase1/weights/best.pt')

# Freeze all except the last 20 layers
num_layers_n = len(list(model_n_ft.model.model.children()))
freeze_layers_n_ft = max(0, num_layers_n - 20)

print(f'Total layers: {num_layers_n}')
print(f'Layers to freeze in FT (unfreezing last 20): {freeze_layers_n_ft}')

# Fine-tune the model
model_n_ft.train(
    data=SPLITTED_DATA_DIR,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    lr0=1e-5,
    optimizer='Adam',
    freeze=freeze_layers_n_ft,
    project='yolo26n_4class',
    name='phase2_finetune',
    exist_ok=True,
    seed=SEED,
    fliplr=0.5,
    flipud=0.5,
    degrees=20.0,
    hsv_v=0.2
)
print('✅ YOLO26n-cls Fine-Tuning Complete.')

### Step 5b: Training History Plots for YOLO26n-cls

In [ ]:
# Load training history CSV
results_csv_n = 'yolo26n_4class/phase2_finetune/results.csv'
if os.path.exists(results_csv_n):
    df_n = pd.read_csv(results_csv_n)
    df_n.columns = [c.strip() for c in df_n.columns]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot Accuracy
    if 'metrics/accuracy_top1' in df_n.columns:
        axes[0].plot(df_n['epoch'], df_n['metrics/accuracy_top1'], label='Val (Top 1)')
    axes[0].set_title('Accuracy — YOLO26n-cls Fine-Tuning')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Plot Loss
    if 'train/loss' in df_n.columns:
        axes[1].plot(df_n['epoch'], df_n['train/loss'], label='Train')
    if 'val/loss' in df_n.columns:
        axes[1].plot(df_n['epoch'], df_n['val/loss'], label='Val')
    axes[1].set_title('Loss — YOLO26n-cls Fine-Tuning')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle('YOLO26n-cls 4-Class Training History', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('YOLO26n-cls training history results.csv not found.')

### Step 6: Evaluate YOLO26n-cls on Test Set

In [ ]:
# Load final fine-tuned model
best_model_n = YOLO('yolo26n_4class/phase2_finetune/weights/best.pt')

# Perform batch prediction on test paths
print('Running prediction on test set...')
results_n = best_model_n.predict(source=list(test_paths), batch=BATCH_SIZE, verbose=False)

y_pred_n = np.array([r.probs.top1 for r in results_n])
y_true = test_labels

acc_n = accuracy_score(y_true, y_pred_n)
print(f'\nYOLO26n-cls Test Accuracy: {acc_n:.4f} ({acc_n*100:.2f}%)\n')
print(classification_report(y_true, y_pred_n, target_names=CLASS_NAMES))

# Confusion Matrix
cm_n = confusion_matrix(y_true, y_pred_n)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_n, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — YOLO26n-cls (All Layers Fine-Tuned)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
# 4. YOLO26s-cls Model Training & Evaluation

In [ ]:
# Step 1 & 2: Load pretrained base model
model_s = YOLO('yolo26s-cls.pt')

# Freeze backbone
num_layers_s = len(list(model_s.model.model.children()))
freeze_layers_s = num_layers_s - 1

print(f'Total layers: {num_layers_s}')
print(f'Layers to freeze (backbone): {freeze_layers_s}')

### Step 4: YOLO26s-cls Phase 1 — Frozen Base Training (`lr0 = 1e-4`)

In [ ]:
# Train model with frozen backbone
model_s.train(
    data=SPLITTED_DATA_DIR,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    lr0=1e-4,
    optimizer='Adam',
    freeze=freeze_layers_s,
    project='yolo26s_4class',
    name='phase1',
    exist_ok=True,
    seed=SEED,
    fliplr=0.5,
    flipud=0.5,
    degrees=20.0,
    hsv_v=0.2
)
print('✅ YOLO26s-cls Phase 1 Complete.')

### Step 5: YOLO26s-cls Phase 2 — Fine-Tuning (Unfreeze Last 20 Layers)

In [ ]:
# Load Phase-1 best weights
model_s_ft = YOLO('yolo26s_4class/phase1/weights/best.pt')

# Freeze all except the last 20 layers
num_layers_s = len(list(model_s_ft.model.model.children()))
freeze_layers_s_ft = max(0, num_layers_s - 20)

print(f'Total layers: {num_layers_s}')
print(f'Layers to freeze in FT (unfreezing last 20): {freeze_layers_s_ft}')

# Fine-tune the model
model_s_ft.train(
    data=SPLITTED_DATA_DIR,
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    lr0=1e-5,
    optimizer='Adam',
    freeze=freeze_layers_s_ft,
    project='yolo26s_4class',
    name='phase2_finetune',
    exist_ok=True,
    seed=SEED,
    fliplr=0.5,
    flipud=0.5,
    degrees=20.0,
    hsv_v=0.2
)
print('✅ YOLO26s-cls Fine-Tuning Complete.')

### Step 5b: Training History Plots for YOLO26s-cls

In [ ]:
# Load training history CSV
results_csv_s = 'yolo26s_4class/phase2_finetune/results.csv'
if os.path.exists(results_csv_s):
    df_s = pd.read_csv(results_csv_s)
    df_s.columns = [c.strip() for c in df_s.columns]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Plot Accuracy
    if 'metrics/accuracy_top1' in df_s.columns:
        axes[0].plot(df_s['epoch'], df_s['metrics/accuracy_top1'], label='Val (Top 1)')
    axes[0].set_title('Accuracy — YOLO26s-cls Fine-Tuning')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # Plot Loss
    if 'train/loss' in df_s.columns:
        axes[1].plot(df_s['epoch'], df_s['train/loss'], label='Train')
    if 'val/loss' in df_s.columns:
        axes[1].plot(df_s['epoch'], df_s['val/loss'], label='Val')
    axes[1].set_title('Loss — YOLO26s-cls Fine-Tuning')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle('YOLO26s-cls 4-Class Training History', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('YOLO26s-cls training history results.csv not found.')

### Step 6: Evaluate YOLO26s-cls on Test Set

In [ ]:
# Load final fine-tuned model
best_model_s = YOLO('yolo26s_4class/phase2_finetune/weights/best.pt')

# Perform batch prediction on test paths
print('Running prediction on test set...')
results_s = best_model_s.predict(source=list(test_paths), batch=BATCH_SIZE, verbose=False)

y_pred_s = np.array([r.probs.top1 for r in results_s])
y_true = test_labels

acc_s = accuracy_score(y_true, y_pred_s)
print(f'\nYOLO26s-cls Test Accuracy: {acc_s:.4f} ({acc_s*100:.2f}%)\n')
print(classification_report(y_true, y_pred_s, target_names=CLASS_NAMES))

# Confusion Matrix
cm_s = confusion_matrix(y_true, y_pred_s)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_s, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — YOLO26s-cls (All Layers Fine-Tuned)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

---
# 5. Results Comparison (Step 7)

Below is a comparison of the YOLO26 classification models evaluated on the test set:

In [ ]:
# Build comparison table for YOLO26 models
yolo_results = [
    {'Model': 'YOLO26n-cls (Nano)', 'Test Accuracy': acc_n, 'Parameters': '1.54M', 'FLOPs': '3.3G'},
    {'Model': 'YOLO26s-cls (Small)', 'Test Accuracy': acc_s, 'Parameters': '6.7M', 'FLOPs': '1.6B'}
]

df_yolo = pd.DataFrame(yolo_results)
print(df_yolo.to_markdown(index=False))